# PointPillars — 3-D Point Cloud Semantic Segmentation

Adapts the **PointPillars** architecture (Lang et al., CVPR 2019) for
per-point semantic segmentation of warehouse LiDAR scans.

**Data layout on Google Drive**
```
training_data/lidar/
  train/scans/{000000.npy, ...}   float32 (N,4)  x y z intensity  (sensor frame)
  train/labels/{000000.npy, ...}  int32   (N,)   0=bg 1=shelf 2=crate 3=forklift
  val/  …same structure…
```

**Architecture**
1. **PillarFeatureNet** — 9-feature augmented points → shared Linear → BN → ReLU → max-pool → 64-d pillar vector  
2. **Scatter** — sparse pillar vectors → dense (64, 160, 160) BEV pseudo-image  
3. **BEV Backbone** — three conv blocks + FPN deconvolutions → (320, 160, 160)  
4. **Segmentation head** — 1×1 convolutions → (4, 160, 160) per-pillar class logits  
5. **Gather** — index BEV logits at each point's pillar → per-point cross-entropy loss  

In [ ]:
# ── 1. Mount Drive and install dependencies ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# PyTorch ships pre-installed on Colab; only extras needed:
!pip install -q scipy

In [ ]:
# ── 2. Imports and configuration ─────────────────────────────────────────────
import os, glob, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

torch.manual_seed(42)
np.random.seed(42)


class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    DATA_ROOT = '/content/drive/MyDrive/training_data/lidar'
    CKPT_DIR  = '/content/drive/MyDrive/training_data/checkpoints/pointpillars'

    # ── Classes ──────────────────────────────────────────────────────────────
    NUM_CLASSES = 4
    CLASS_NAMES = ['background', 'shelf', 'crate', 'forklift']

    # ── Pillar geometry (sensor frame: +x fwd, +y left, +z up) ───────────────
    X_MIN, X_MAX = -20.0, 20.0   # metres — covers full 20 m LiDAR range
    Y_MIN, Y_MAX = -20.0, 20.0
    Z_MIN, Z_MAX =  -0.5,  3.0   # floor below sensor to ~ceiling
    VX,    VY    =  0.25,  0.25  # pillar footprint (m)
    MAX_PTS_PER_PILLAR = 32
    MAX_PILLARS        = 6000    # sparse cap on occupied pillars per scan

    # Derived BEV grid
    BEV_H = int((X_MAX - X_MIN) / VX)   # 160
    BEV_W = int((Y_MAX - Y_MIN) / VY)   # 160

    # ── Model ────────────────────────────────────────────────────────────────
    PILLAR_FEAT = 64

    # ── Training ─────────────────────────────────────────────────────────────
    BATCH_SIZE   = 4
    NUM_EPOCHS   = 60
    LR           = 1e-3
    WEIGHT_DECAY = 1e-4
    LR_MILESTONES = [30, 50]
    MAX_PTS      = 20_000        # subsample scans larger than this

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


cfg = Config()
os.makedirs(cfg.CKPT_DIR, exist_ok=True)
print(f'Device : {cfg.DEVICE}')
print(f'BEV    : {cfg.BEV_H} × {cfg.BEV_W}  pillar {cfg.VX} m')

In [ ]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────
class LiDARSegDataset(Dataset):
    """Loads pre-saved LiDAR scans and per-point labels."""

    def __init__(self, split='train', augment=False):
        self.augment = augment and (split == 'train')
        scan_dir = os.path.join(cfg.DATA_ROOT, split, 'scans')
        lbl_dir  = os.path.join(cfg.DATA_ROOT, split, 'labels')
        self.scans  = sorted(glob.glob(os.path.join(scan_dir,  '*.npy')))
        self.labels = [os.path.join(lbl_dir, os.path.basename(f))
                       for f in self.scans]
        print(f'[{split}] {len(self.scans)} scans')

    def __len__(self):
        return len(self.scans)

    def __getitem__(self, idx):
        pts = np.load(self.scans[idx]).astype(np.float32)   # (N, 4)
        lbl = np.load(self.labels[idx]).astype(np.int64)    # (N,)

        # Spatial crop to configured range
        keep = (
            (pts[:, 0] >= cfg.X_MIN) & (pts[:, 0] < cfg.X_MAX) &
            (pts[:, 1] >= cfg.Y_MIN) & (pts[:, 1] < cfg.Y_MAX) &
            (pts[:, 2] >= cfg.Z_MIN) & (pts[:, 2] < cfg.Z_MAX)
        )
        pts, lbl = pts[keep], lbl[keep]

        # Random subsample
        if len(pts) > cfg.MAX_PTS:
            sel = np.random.choice(len(pts), cfg.MAX_PTS, replace=False)
            pts, lbl = pts[sel], lbl[sel]

        # Augmentation: random yaw rotation ± π
        if self.augment and len(pts) > 0:
            angle  = np.random.uniform(-np.pi, np.pi)
            c, s   = np.cos(angle), np.sin(angle)
            xy     = pts[:, :2] @ np.array([[c, s], [-s, c]], np.float32)
            pts    = np.concatenate([xy, pts[:, 2:]], axis=1)
            # Intensity jitter
            pts[:, 3] = np.clip(pts[:, 3] + np.random.normal(0, 0.01, len(pts)), 0, 1)

        return pts, lbl   # variable-length; collate handles batching


def collate_var(batch):
    """Return list of arrays — no stacking needed for variable-size clouds."""
    pts_list = [b[0] for b in batch]
    lbl_list = [b[1] for b in batch]
    return pts_list, lbl_list


train_ds = LiDARSegDataset('train', augment=True)
val_ds   = LiDARSegDataset('val',   augment=False)
train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                      shuffle=True,  collate_fn=collate_var, num_workers=2,
                      pin_memory=(cfg.DEVICE == 'cuda'))
val_dl   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                      shuffle=False, collate_fn=collate_var, num_workers=2,
                      pin_memory=(cfg.DEVICE == 'cuda'))
print(f'Train batches: {len(train_dl)}   Val batches: {len(val_dl)}')

In [ ]:
# ── 4. Pillarization ──────────────────────────────────────────────────────────
def pillarize(pts_list):
    """
    Convert a batch of point clouds to the sparse pillar representation
    described in Lang et al. (2019), Section 2.1.

    Each point is augmented with 9 features:
      [x, y, z, r,  Δx, Δy, Δz,  x_p, y_p]
       └── raw ──┘  └ from mean ┘  └ pillar ctr ┘

    Returns
    -------
    pillars  : (B, P, K, 9)  float32
    coords   : (B, P, 2)     int32   BEV indices (xi, yi)
    n_pil    : (B,)          int     valid pillars per sample
    pt2pil   : list of (N_b,) int32  pillar slot for each point (-1 = dropped)
    """
    B  = len(pts_list)
    P  = cfg.MAX_PILLARS
    K  = cfg.MAX_PTS_PER_PILLAR

    pillars = np.zeros((B, P, K, 9), np.float32)
    coords  = np.zeros((B, P, 2),   np.int32)
    n_pil   = np.zeros(B,           np.int32)
    pt2pil  = []

    for b, pts in enumerate(pts_list):
        if len(pts) == 0:
            pt2pil.append(np.empty(0, np.int32))
            continue

        N  = len(pts)
        xi = np.clip(np.floor((pts[:, 0] - cfg.X_MIN) / cfg.VX).astype(np.int32),
                     0, cfg.BEV_H - 1)
        yi = np.clip(np.floor((pts[:, 1] - cfg.Y_MIN) / cfg.VY).astype(np.int32),
                     0, cfg.BEV_W - 1)
        key = xi.astype(np.int64) * cfg.BEV_W + yi

        # Sort by pillar key for contiguous grouping
        order  = np.argsort(key, kind='mergesort')
        key_s  = key[order]
        pts_s  = pts[order]

        # Boundary of each pillar group
        bounds = np.where(np.diff(key_s, prepend=key_s[0] - 1) != 0)[0]
        ukeys  = key_s[bounds]
        n_keep = min(len(ukeys), P)
        p2p    = np.full(N, -1, np.int32)

        for pi in range(n_keep):
            s  = bounds[pi]
            e  = bounds[pi + 1] if pi + 1 < len(bounds) else N
            n  = min(e - s, K)
            orig_idx = order[s: s + n]       # indices into original pts
            p2p[orig_idx] = pi

            pts_pi = pts_s[s: s + n]         # (n, 4)
            mean   = pts_pi[:, :3].mean(0)
            xi_p   = int(ukeys[pi]) // cfg.BEV_W
            yi_p   = int(ukeys[pi]) %  cfg.BEV_W
            xp     = xi_p * cfg.VX + cfg.X_MIN + cfg.VX / 2
            yp     = yi_p * cfg.VY + cfg.Y_MIN + cfg.VY / 2

            feat = np.empty((n, 9), np.float32)
            feat[:, :4]  = pts_pi
            feat[:, 4:7] = pts_pi[:, :3] - mean   # Δx, Δy, Δz
            feat[:, 7]   = xp
            feat[:, 8]   = yp

            pillars[b, pi, :n] = feat
            coords[b, pi]      = [xi_p, yi_p]

        n_pil[b] = n_keep
        pt2pil.append(p2p)

    return (
        torch.from_numpy(pillars),   # (B, P, K, 9)
        torch.from_numpy(coords),    # (B, P, 2)
        n_pil,                       # (B,)  numpy int
        pt2pil,                      # list of (N_b,) numpy int32
    )


# Quick sanity check
_pts, _lbl = train_ds[0]
_pil, _coo, _n, _p2p = pillarize([_pts])
print(f'Scan 0: {len(_pts)} pts → {_n[0]} pillars  '
      f'pillars shape {_pil.shape}  dropped {(_p2p[0] == -1).sum()}')

In [ ]:
# ── 5. Model: PillarFeatureNet ────────────────────────────────────────────────
class PillarFeatureNet(nn.Module):
    """
    Shared linear layer + BN + ReLU applied to every point in every pillar,
    followed by channel-wise max-pool over the point axis.
    Output: one 64-d feature vector per pillar.
    """
    def __init__(self, in_ch=9, out_ch=64):
        super().__init__()
        self.lin = nn.Linear(in_ch, out_ch, bias=False)
        self.bn  = nn.BatchNorm1d(out_ch)

    def forward(self, pillars):   # (B*P, K, in_ch)
        BP, K, C = pillars.shape
        x = self.lin(pillars.reshape(BP * K, C))   # (BP*K, out_ch)
        x = F.relu(self.bn(x), inplace=True)
        x = x.view(BP, K, -1).max(dim=1)[0]        # (BP, out_ch)
        return x


# ── 6. Model: BEV Backbone ───────────────────────────────────────────────────
class _ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, depth=4):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        for _ in range(depth - 1):
            layers += [nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


class BEVBackbone(nn.Module):
    """
    Three-block 2-D CNN with FPN-style upsampling (Lang et al. Table 1).
    Outputs a 320-channel feature map at the original BEV resolution.
    """
    def __init__(self, in_ch=64):
        super().__init__()
        self.b1 = _ConvBlock(in_ch, 64,  stride=1, depth=4)   # (B,64,H,W)
        self.b2 = _ConvBlock(64,   128,  stride=2, depth=6)   # (B,128,H/2,W/2)
        self.b3 = _ConvBlock(128,  256,  stride=2, depth=6)   # (B,256,H/4,W/4)

        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=4, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))

        self.out_ch = 64 + 128 + 128   # 320

    def forward(self, x):
        f1 = self.b1(x)
        f2 = self.b2(f1)
        f3 = self.b3(f2)
        return torch.cat([f1, self.up2(f2), self.up3(f3)], dim=1)

In [ ]:
# ── 7. Full model: PointPillarsSeg ────────────────────────────────────────────
class PointPillarsSeg(nn.Module):
    """
    End-to-end PointPillars for semantic segmentation.

    forward() → logits_bev (B, C, H, W)  per-pillar class logits.
    Per-point logits are obtained by indexing logits_bev at each point's
    (xi, yi) pillar coordinate — no additional parameters needed.
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.pfn     = PillarFeatureNet(in_ch=9, out_ch=cfg.PILLAR_FEAT)
        self.backbone = BEVBackbone(in_ch=cfg.PILLAR_FEAT)
        self.head    = nn.Sequential(
            nn.Conv2d(self.backbone.out_ch, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, num_classes, 1),
        )
        self.num_classes = num_classes
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, pillars, coords, n_pil):
        """
        pillars : (B, P, K, 9)   on device
        coords  : (B, P, 2)      int, BEV indices
        n_pil   : (B,)           valid pillar count
        """
        B, P, K, C = pillars.shape
        H, W = cfg.BEV_H, cfg.BEV_W

        # PillarFeatureNet → (B, P, 64)
        pf = self.pfn(pillars.view(B * P, K, C)).view(B, P, cfg.PILLAR_FEAT)

        # Scatter to dense BEV canvas
        bev = pillars.new_zeros(B, cfg.PILLAR_FEAT, H, W)
        for b in range(B):
            n  = int(n_pil[b])
            xi = coords[b, :n, 0].long()
            yi = coords[b, :n, 1].long()
            bev[b, :, xi, yi] = pf[b, :n].T

        feat       = self.backbone(bev)   # (B, 320, H, W)
        logits_bev = self.head(feat)      # (B, C, H, W)
        return logits_bev


# Verify forward pass
_model = PointPillarsSeg().to(cfg.DEVICE)
_pil_t = _pil.to(cfg.DEVICE)
_coo_t = _coo.to(cfg.DEVICE)
with torch.no_grad():
    _out = _model(_pil_t, _coo_t, _n)
print(f'logits_bev shape: {_out.shape}')    # expect (1, 4, 160, 160)
total_params = sum(p.numel() for p in _model.parameters())
print(f'Parameters: {total_params:,}')
del _model, _pil_t, _coo_t, _out

In [ ]:
# ── 8. Loss and metrics ───────────────────────────────────────────────────────
def estimate_class_weights(dataset, n_sample=400):
    """Inverse-frequency class weights from a random subset."""
    counts = np.zeros(cfg.NUM_CLASSES, np.int64)
    for i in np.random.choice(len(dataset), min(n_sample, len(dataset)), replace=False):
        _, lbl = dataset[i]
        for c in range(cfg.NUM_CLASSES):
            counts[c] += int((lbl == c).sum())
    counts  = np.maximum(counts, 1)
    weights = 1.0 / counts.astype(np.float64)
    weights = weights / weights.sum() * cfg.NUM_CLASSES   # normalise to mean=1
    return torch.tensor(weights, dtype=torch.float32)


print('Estimating class weights…')
class_weights = estimate_class_weights(train_ds).to(cfg.DEVICE)
print('Class weights:', dict(zip(cfg.CLASS_NAMES,
                                  class_weights.cpu().numpy().round(3))))
criterion = nn.CrossEntropyLoss(weight=class_weights)


def per_class_iou(pred_np, true_np):
    """Returns list of IoU values (NaN when class absent)."""
    iou = []
    for c in range(cfg.NUM_CLASSES):
        tp = int(((pred_np == c) & (true_np == c)).sum())
        fp = int(((pred_np == c) & (true_np != c)).sum())
        fn = int(((pred_np != c) & (true_np == c)).sum())
        d  = tp + fp + fn
        iou.append(tp / d if d > 0 else float('nan'))
    return iou

In [ ]:
# ── 9. Train / evaluate functions ─────────────────────────────────────────────
def gather_point_logits(logits_bev, coords, n_pil, pt2pil, lbl_list=None):
    """
    Index logits_bev at each point's pillar coordinate.

    Returns
    -------
    all_logits : (M, C) stacked per-point logits (M = valid points in batch)
    all_labels : (M,)   matching ground-truth labels  (or None)
    """
    all_logits, all_labels = [], []
    B = logits_bev.shape[0]
    for b in range(B):
        p2p   = pt2pil[b]            # (N_b,) numpy
        valid = p2p >= 0
        if not valid.any():
            continue
        xi = torch.from_numpy(coords[b, p2p[valid], 0].astype(np.int64)).to(logits_bev.device)
        yi = torch.from_numpy(coords[b, p2p[valid], 1].astype(np.int64)).to(logits_bev.device)
        all_logits.append(logits_bev[b, :, xi, yi].T)   # (n_valid, C)
        if lbl_list is not None:
            all_labels.append(
                torch.from_numpy(lbl_list[b][valid]).to(logits_bev.device))
    if not all_logits:
        return None, None
    return torch.cat(all_logits), (torch.cat(all_labels) if all_labels else None)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for pts_list, lbl_list in loader:
        pil, coo, n_pil, p2p = pillarize(pts_list)
        pil = pil.to(cfg.DEVICE)
        coo = coo.to(cfg.DEVICE)

        optimizer.zero_grad()
        logits_bev        = model(pil, coo, n_pil)
        logits_pt, lbl_pt = gather_point_logits(logits_bev, coo.cpu().numpy(),
                                                 n_pil, p2p, lbl_list)
        if logits_pt is None:
            continue
        loss = criterion(logits_pt, lbl_pt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 35.0)
        optimizer.step()
        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    for pts_list, lbl_list in loader:
        pil, coo, n_pil, p2p = pillarize(pts_list)
        pil = pil.to(cfg.DEVICE)
        coo = coo.to(cfg.DEVICE)
        logits_bev        = model(pil, coo, n_pil)
        logits_pt, lbl_pt = gather_point_logits(logits_bev, coo.cpu().numpy(),
                                                 n_pil, p2p, lbl_list)
        if logits_pt is None:
            continue
        all_pred.append(logits_pt.argmax(1).cpu().numpy())
        all_true.append(lbl_pt.cpu().numpy())

    if not all_pred:
        return [float('nan')] * cfg.NUM_CLASSES, float('nan')
    pred_np = np.concatenate(all_pred)
    true_np = np.concatenate(all_true)
    iou     = per_class_iou(pred_np, true_np)
    valid   = [v for v in iou if not math.isnan(v)]
    miou    = sum(valid) / len(valid) if valid else float('nan')
    return iou, miou

In [ ]:
# ── 10. Training loop ─────────────────────────────────────────────────────────
model     = PointPillarsSeg(num_classes=cfg.NUM_CLASSES).to(cfg.DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR,
                              weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=cfg.LR_MILESTONES, gamma=0.1)

history   = {'loss': [], 'miou': [], 'iou': []}
best_miou = 0.0

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    loss = train_epoch(model, train_dl, optimizer)
    scheduler.step()
    history['loss'].append(loss)

    if epoch % 5 == 0 or epoch == 1:
        iou_list, miou = evaluate(model, val_dl)
        history['miou'].append((epoch, miou))
        history['iou'].append((epoch, iou_list))

        iou_str = '  '.join(
            f'{cfg.CLASS_NAMES[c]}={v:.3f}' if not math.isnan(v) else f'{cfg.CLASS_NAMES[c]}=--'
            for c, v in enumerate(iou_list))
        print(f'Ep {epoch:3d}/{cfg.NUM_EPOCHS}  loss={loss:.4f}  '
              f'mIoU={miou:.4f}  [{iou_str}]')

        if miou > best_miou:
            best_miou = miou
            torch.save({'epoch': epoch,
                        'state_dict': model.state_dict(),
                        'miou': miou, 'iou': iou_list},
                       os.path.join(cfg.CKPT_DIR, 'best.pt'))
            print(f'  ✓ best checkpoint saved (mIoU={miou:.4f})')

        if epoch % 10 == 0:
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict()},
                       os.path.join(cfg.CKPT_DIR, f'epoch_{epoch:03d}.pt'))
            print(f'  ✓ periodic checkpoint saved (epoch {epoch})')

print(f'\nDone.  Best val mIoU = {best_miou:.4f}')

In [ ]:
# ── 11. Evaluation and visualisation ─────────────────────────────────────────
# Load best checkpoint
ckpt = torch.load(os.path.join(cfg.CKPT_DIR, 'best.pt'), map_location=cfg.DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f"Best checkpoint — epoch {ckpt['epoch']}  mIoU={ckpt['miou']:.4f}")

# Training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['loss'])
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training loss'); axes[0].grid(True)

ep_vals  = [e for e, _ in history['miou']]
miou_vals = [m for _, m in history['miou']]
axes[1].plot(ep_vals, miou_vals, 'g-o', ms=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation mIoU'); axes[1].grid(True)
plt.tight_layout(); plt.show()

# Final per-class IoU table
iou_list, miou = evaluate(model, val_dl)
print('\n── Validation IoU ──────────────────────')
print(f'{"Class":<14} {"IoU":>8}')
print('─' * 24)
for c, v in enumerate(iou_list):
    s = f'{v:.4f}' if not math.isnan(v) else '    --'
    print(f'{cfg.CLASS_NAMES[c]:<14} {s:>8}')
print(f'{"mIoU":<14} {miou:.4f}')

# Qualitative BEV plot — one validation scan
model.eval()
pts_list_v, lbl_list_v = next(iter(val_dl))
pil_v, coo_v, n_pil_v, p2p_v = pillarize(pts_list_v)
with torch.no_grad():
    logits_v = model(pil_v.to(cfg.DEVICE), coo_v.to(cfg.DEVICE), n_pil_v)
logits_pt_v, _ = gather_point_logits(logits_v, coo_v.numpy(), n_pil_v, p2p_v)

COLORS = np.array([[0.45,0.45,0.45], [0.20,0.70,0.20],
                    [0.90,0.15,0.15], [0.10,0.40,0.90]])
b = 0
pts  = pts_list_v[b]
true = lbl_list_v[b]
valid = p2p_v[b] >= 0
pts_v  = pts[valid]
true_v = true[valid]

# gather predictions for sample b
with torch.no_grad():
    xi_b = torch.from_numpy(coo_v[b, p2p_v[b][valid], 0].astype(np.int64))
    yi_b = torch.from_numpy(coo_v[b, p2p_v[b][valid], 1].astype(np.int64))
    pred_v = logits_v[b, :, xi_b.to(cfg.DEVICE), yi_b.to(cfg.DEVICE)].argmax(0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, labels, title in zip(axes, [true_v, pred_v], ['Ground Truth', 'Predicted']):
    c = COLORS[np.clip(labels, 0, 3)]
    ax.scatter(pts_v[:, 1], pts_v[:, 0], c=c, s=1.0, alpha=0.85, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Y — left (m)'); ax.set_ylabel('X — forward (m)')
    ax.set_xlim(cfg.Y_MIN, cfg.Y_MAX); ax.set_ylim(cfg.X_MIN, cfg.X_MAX)
    legend = [Patch(color=COLORS[c], label=cfg.CLASS_NAMES[c])
               for c in range(cfg.NUM_CLASSES)]
    ax.legend(handles=legend, loc='upper right')
plt.suptitle("PointPillars Segmentation — Bird's Eye View", fontsize=14)
plt.tight_layout(); plt.show()